# Greenhouse Slug Discovery

This notebook tests candidate Greenhouse board slugs against:

`https://boards-api.greenhouse.io/v1/boards/{slug}/jobs`

A slug is considered **working** when the API responds `200 OK`.

## Is this possible?

Yes. Greenhouse board URLs are public and can be probed by slug.

- `200` means the board exists and is reachable.
- `404` usually means no board for that slug.

> Note: this cannot guarantee a complete list of *all* companies using Greenhouse.
> It only validates the candidate slugs you provide.

In [1]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import pandas as pd


In [2]:
# --- Configuration ---
BASE_URL = "https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 12
SLEEP_JITTER_SEC = (0.0, 0.15)  # polite jitter between calls in each worker

# Seed list from the app's current curated companies.
seed_slugs = [
    "airbnb", "amplitude", "anthropic", "asana", "benchling", "brex", "canva", "cloudflare",
    "coinbase", "datadog", "databricks", "discord", "doordash", "dropbox", "duolingo",
    "figma", "fivetran", "gitlab", "gusto", "hashicorp", "hubspot", "intercom", "lattice",
    "launchdarkly", "linear", "loom", "lyft", "mercury", "miro", "mixpanel", "mongodb",
    "notion", "pagerduty", "palantir", "pinterest", "plaid", "ramp", "reddit", "retool",
    "rippling", "robinhood", "scale", "segment", "snowflake", "stripe", "twilio", "verkada",
    "zendesk", "zoom"
]

# Add your own guesses here (lowercase slug format):
extra_candidates = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/companies.csv")["company"].dropna().tolist()

candidates = sorted(set(seed_slugs + extra_candidates))
print(f"Total candidate slugs: {len(candidates)}")

Total candidate slugs: 1047


In [3]:
def probe_slug(slug: str) -> dict:
    url = BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        # jitter to be polite when running high concurrency
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(r.json().get("jobs", [])) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_slugs(slugs: Iterable[str]) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [12]:
results = scan_slugs(candidates)
results.head(200)

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,2U,200,True,29.0,https://boards-api.greenhouse.io/v1/boards/2U/...,273.7
1,6sense,200,True,29.0,https://boards-api.greenhouse.io/v1/boards/6se...,202.0
2,ASG,200,True,2.0,https://boards-api.greenhouse.io/v1/boards/ASG...,218.5
3,Acquia,200,True,10.0,https://boards-api.greenhouse.io/v1/boards/Acq...,125.5
4,Affirm,200,True,163.0,https://boards-api.greenhouse.io/v1/boards/Aff...,131.4
...,...,...,...,...,...,...
195,Amazon Pay,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ama...,150.8
196,Amazon Web Services,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ama...,139.9
197,Amdocs,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Amd...,138.0
198,Amelia,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ame...,140.0


In [15]:
results["url"].tail(200)

847     https://boards-api.greenhouse.io/v1/boards/Spl...
848     https://boards-api.greenhouse.io/v1/boards/Spo...
849     https://boards-api.greenhouse.io/v1/boards/Spr...
850     https://boards-api.greenhouse.io/v1/boards/Spr...
851     https://boards-api.greenhouse.io/v1/boards/Spr...
                              ...                        
1042    https://boards-api.greenhouse.io/v1/boards/sno...
1043    https://boards-api.greenhouse.io/v1/boards/sym...
1044    https://boards-api.greenhouse.io/v1/boards/tes...
1045    https://boards-api.greenhouse.io/v1/boards/zen...
1046    https://boards-api.greenhouse.io/v1/boards/zoo...
Name: url, Length: 200, dtype: object

In [5]:
summary = (
    results.assign(status_group=results["status_code"].fillna(-1))
    .groupby("status_group", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("status_group")
)
summary

,status_group,count
0,200,154
1,404,893


In [ ]:
working = results[results["ok"]].copy()
print(f"Working slugs: {len(working)}")
working[["slug", "jobs_count", "status_code", "elapsed_ms"]].head
(50)

Working slugs: 154


,slug,jobs_count,status_code,elapsed_ms
104,Toast,275.0,200,152.1
105,Transcarent,31.0,200,163.8
106,Tripadvisor,63.0,200,137.8
107,Turing,28.0,200,177.0
108,Twilio,127.0,200,173.0
109,Upgrade,18.0,200,132.3
110,Upstart,70.0,200,141.3
111,Vonage,80.0,200,144.4
112,Waymo,419.0,200,147.2
113,Wrike,49.0,200,141.0


In [7]:
# Save results for later use.
output_csv = "greenhouse_slug_scan_results.csv"
results.sort_values(["ok", "slug"], ascending=[False, True]).to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

Saved: greenhouse_slug_scan_results.csv


---

## Ashby API Check

Testing the same candidate slugs against the Ashby public job board API:

`https://api.ashbyhq.com/posting-api/job-board/{slug}`

- `200` means the company has an Ashby job board at that slug.
- `404` / other means no board found.

> Slugs are case-sensitive in Ashby — this first pass uses lowercase (same as Greenhouse slugs).

In [9]:
ASHBY_BASE_URL = "https://api.ashbyhq.com/posting-api/job-board/{slug}"

def probe_ashby_slug(slug: str) -> dict:
    url = ASHBY_BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        jobs = []
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobs", [])

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(jobs) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_ashby_slugs(slugs) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_ashby_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [10]:
ashby_results = scan_ashby_slugs(candidates)

ashby_working = ashby_results[ashby_results["ok"]].copy()
print(f"Working Ashby slugs: {len(ashby_working)} / {len(candidates)}")
ashby_results

Working Ashby slugs: 40 / 1047


,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Applied Intuition,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,279.9
1,Boomi,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,277.2
2,Centerfield,200,True,20.0,https://api.ashbyhq.com/posting-api/job-board/...,590.9
3,Confluent,200,True,58.0,https://api.ashbyhq.com/posting-api/job-board/...,666.4
4,Crusoe,200,True,306.0,https://api.ashbyhq.com/posting-api/job-board/...,4737.4
...,...,...,...,...,...,...
1042,test IO,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,324.7
1043,twilio,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,269.7
1044,xAI,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/xAI,313.1
1045,zendesk,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,465.0


In [21]:
ashby_working.head(50)

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Applied Intuition,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,279.9
1,Boomi,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,277.2
2,Centerfield,200,True,20.0,https://api.ashbyhq.com/posting-api/job-board/...,590.9
3,Confluent,200,True,58.0,https://api.ashbyhq.com/posting-api/job-board/...,666.4
4,Crusoe,200,True,306.0,https://api.ashbyhq.com/posting-api/job-board/...,4737.4
5,Dandy,200,True,143.0,https://api.ashbyhq.com/posting-api/job-board/...,2086.1
6,Delinea,200,True,105.0,https://api.ashbyhq.com/posting-api/job-board/...,1546.8
7,Everbridge,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,332.2
8,Handshake,200,True,89.0,https://api.ashbyhq.com/posting-api/job-board/...,1634.3
9,Harmonic,200,True,7.0,https://api.ashbyhq.com/posting-api/job-board/...,339.9
